In [1]:
!pip install boto3 requests

   ---------------------------------------- 0.0/14.7 MB ? eta -:--:--
   -- ------------------------------------- 1.0/14.7 MB 5.0 MB/s eta 0:00:03
   ---- ----------------------------------- 1.6/14.7 MB 3.7 MB/s eta 0:00:04
   ----- ---------------------------------- 2.1/14.7 MB 3.7 MB/s eta 0:00:04
   ------- -------------------------------- 2.9/14.7 MB 3.5 MB/s eta 0:00:04
   --------- ------------------------------ 3.4/14.7 MB 3.3 MB/s eta 0:00:04
   ------------ --------------------------- 4.5/14.7 MB 3.5 MB/s eta 0:00:03
   --------------- ------------------------ 5.8/14.7 MB 3.9 MB/s eta 0:00:03
   ------------------ --------------------- 6.8/14.7 MB 4.1 MB/s eta 0:00:02
   ---------------------- ----------------- 8.1/14.7 MB 4.3 MB/s eta 0:00:02
   ------------------------- -------------- 9.4/14.7 MB 4.4 MB/s eta 0:00:02
   ---------------------------- ----------- 10.5/14.7 MB 4.5 MB/s eta 0:00:01
   ------------------------------- -------- 11.5/14.7 MB 4.5 MB/s eta 0:00:01
   -

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.0 requires botocore<1.40.50,>=1.40.46, but you have botocore 1.42.68 which is incompatible.


In [2]:
import os

os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key_here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key_here'
os.environ['AWS_REGION'] = 'us-east-1'

print('Credentials set!')

Credentials set!


In [2]:
import boto3, base64, json

def analyze_image(image_bytes, media_type='image/jpeg'):
    aws_key = os.getenv('AWS_ACCESS_KEY_ID')
    aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')

    if not aws_key or aws_key == 'your_access_key_here':
        return 'pothole detected on road surface (mock response)'

    bedrock = boto3.client(
        service_name='bedrock-runtime',
        region_name='us-east-1',
        aws_access_key_id=aws_key,
        aws_secret_access_key=aws_secret,
    )
    image_b64 = base64.b64encode(image_bytes).decode('utf-8')
    body = {
        'messages': [{'role': 'user', 'content': [
            {'image': {'format': media_type.split('/')[-1], 'source': {'bytes': image_b64}}},
            {'text': 'What city problem do you see? Reply in one sentence.'}
        ]}],
        'inferenceConfig': {'maxTokens': 200}
    }
    response = bedrock.invoke_model(
        modelId='amazon.nova-lite-v1:0',
        body=json.dumps(body),
        contentType='application/json',
        accept='application/json'
    )
    result = json.loads(response['body'].read())
    return result['output']['message']['content'][0]['text']

print('Function ready!')

Function ready!


In [ ]:
# Change pothole.jpg to your image file name
image_path = 'pothole.jpg'

with open(image_path, 'rb') as f:
    image_bytes = f.read()

result = analyze_image(image_bytes)
print('Nova says:', result)

In [ ]:
DEPARTMENT_MAP = {
    'pothole': 'Road Maintenance',
    'garbage': 'Waste Management',
    'trash': 'Waste Management',
    'streetlight': 'Electricity Department',
    'broken light': 'Electricity Department',
}

def classify_issue(text):
    for keyword, department in DEPARTMENT_MAP.items():
        if keyword in text.lower():
            return keyword, department
    return 'unknown_issue', 'General Maintenance'

issue, dept = classify_issue(result)
print(f'Issue      : {issue}')
print(f'Department : {dept}')